In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'
import daq
import spectrum as sp
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = daq.run_log()

df_ee_ae_s_filter = lambda ee, ae, s: df.loc[(df['excitation_energy_ev']==ee)&(df['arm_energy_ev']==ae)&(df['sample']==s)]

from droplet_temperature import DropletCooling
cool = DropletCooling(diameter_um=17.9, spacing_um=153,
                      frequency_hz=120e3, pressure_pa=0.015, max_time=11e-2)

def df_y_filter(ee, ae, s, y, atol=1e-3):
    df = df_ee_ae_s_filter(ee, ae, s)
    return df.loc[(df['y'] - y).abs() < atol]

In [ ]:
df

## Parameter comparison

In [ ]:
runs_at_energy = df_ee_ae_s_filter(548.96, 525, "droplets")
grouped_runs = runs_at_energy.groupby(runs_at_energy['y'])
colors = plt.get_cmap("viridis")(np.linspace(0, 0.9, len(grouped_runs)))
for idx, (name, group) in enumerate(grouped_runs):
    runs_in_group = group['acq'].to_numpy()
    acqs = [daq.load_acq(a, calibrated=True).binned(8) for a in runs_in_group]
    acq = sp.combine(acqs).smooth(50//8).normalised(energy_range=(526, 529))
    l = f"T={cool.temperature(acq.meta['parts'][0]['a_mp1_y']):.0f}K @ E={acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    acq.plot(label=l, color=colors[idx])
plt.legend()
plt.xlim(526, 529)
plt.ylabel("Intensity (a.u.)")
plt.xlabel(r"Energy (eV)")
plt.show()

In [ ]:
runs_1 = df_y_filter(548.96, 525, "droplets", 4).acq.to_numpy()
runs_2 = df_y_filter(548.96, 525, "droplets", 40).acq.to_numpy()
total_runs = [runs_1, runs_2]
for idx, runs_at_y in enumerate(total_runs):
    acqs = [daq.load_acq(a, calibrated=True).binned(8) for a in runs_at_y]
    acq = sp.combine(acqs).smooth(50//8).normalised(energy_range=(526, 529))
    # l = f"Excitation Energy = {acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    l = f"T={cool.temperature(acq.meta['parts'][0]['a_mp1_y']):.0f}K @ E={acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    acq.plot(label=l)
    print(acq.meta['counts'])
plt.legend()
# plt.ylim(0, 0.5)
plt.xlim(526, 529)
plt.ylabel("Intensity (a.u.)")
plt.xlabel(r"Energy (eV)")
plt.show()

In [ ]:
total_runs = [
    df_y_filter(533.96, 525, "droplets", 4).acq.to_numpy(),
    df_y_filter(533.96, 525, "droplets", 20).acq.to_numpy(),
    df_y_filter(533.96, 525, "droplets", 40).acq.to_numpy()
]
# colors = plt.get_cmap("viridis")(np.linspace(0, 0.9, len(total_runs)))
for idx, runs_at_y in enumerate(total_runs):
    acqs = [daq.load_acq(a, calibrated=True, dark_correct=True).binned(4) for a in runs_at_y]
    acq = sp.combine(acqs).normalised().smooth_gauss(0.06)
    l = f"T={cool.temperature(acq.meta['parts'][0]['a_mp1_y']):.0f}K @ E={acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    acq.plot(label=l)
    print(acq.meta['counts'])
plt.legend()
# plt.ylim(0, 0.5)
plt.xlim(526, 529)
plt.ylabel("Intensity (a.u.)")
plt.xlabel(r"Energy (eV)")
plt.show()

In [ ]:
total_runs = [
    df_y_filter(538.96, 525, "droplets", 4).acq.to_numpy(),
    # df_y_filter(538.96, 525, "droplets", 20).acq.to_numpy(),
    df_y_filter(538.96, 525, "droplets", 40).acq.to_numpy()
]
# colors = plt.get_cmap("viridis")(np.linspace(0, 0.9, len(total_runs)))
for idx, runs_at_y in enumerate(total_runs):
    acqs = [daq.load_acq(a, calibrated=True, dark_correct=True).binned(4) for a in runs_at_y]
    acq = sp.combine(acqs).normalised().smooth_gauss(0.06)
    l = f"T={cool.temperature(acq.meta['parts'][0]['a_mp1_y']):.0f}K @ E={acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    acq.plot(label=l)
    print(acq.meta['counts'])
plt.legend()
# plt.ylim(0, 0.5)
plt.xlim(526, 529)
plt.ylabel("Intensity (a.u.)")
plt.xlabel(r"Energy (eV)")
plt.show()

## Combination comparison

In [ ]:
import numpy as np

ACQS, LO, HI = (df_y_filter(533.96, 525, "droplets", 40).acq.to_numpy()[:-1]), 526, 529
runs = [daq.load_acq(a, calibrated=True, dark_correct=False).binned(4) for a in ACQS]
total = sp.combine(runs)

def _window(s: Spectrum, lo, hi) -> np.ndarray:
    return (s.energy >= (-np.inf if lo is None else lo)) & (s.energy <= (np.inf if hi is None else hi))

fig, (ax, axr) = plt.subplots(2, 1, sharex=True, height_ratios=[3, 1])
for run in runs:
    keep = _window(run, 526, 529)
    area = np.trapezoid(run.intensity, x=run.energy)
    y, dy = run.intensity / area, np.sqrt(np.abs(run.intensity).clip(1)) / area
    ax.fill_between(run.energy, y - dy, y + dy, alpha=0.4, label=f"acq: {run.meta['acq']}")
total.normalised().plot(ax, color="k", label="combined")
ax.set_ylabel("Intensity (a.u.)")
ax.legend()

for chi2, pull in sp.compare(runs, lo=LO, hi=HI):
    pull.plot(axr, lw=0.8)
    print(f"acq {pull.meta['acq']}: chi2_red {chi2:.2f}")
axr.axhspan(-1, 1, color="0.85")
axr.set_ylabel(r"pull ($\sigma$)")
axr.set_xlabel("Energy (eV)")
axr.set_xlim(LO, HI)
plt.show()

have = total.meta["acquisition_time"]
for target in (0.10, 0.05, 0.03):
    n = sp.runs_needed(runs, target, lo=LO, hi=HI)
    t = sp.time_needed(runs, target, lo=LO, hi=HI)
    print(f"Below {target:.0%} error per point: {n:.1f} runs, {t / 3600:.1f} h in total, {max(0, t - have) / 3600:.1f} h more")